# 09 Threshold Sensitivity Analysis

This notebook extends the validation-only positive-relation threshold search
for the selected RE model. It does not retrain the model.

The decision rule is selected exclusively on validation positive macro F1.
The chosen threshold is then applied once to the existing test probabilities.
Raw report text is never written to the output files.


In [ ]:
from __future__ import annotations

import inspect
import json
import os
import shutil
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
INTERIM_DIR = RUN_ROOT / "interim"
RESULTS_DIR = RUN_ROOT / "results"
PREDICTIONS_DIR = RUN_ROOT / "predictions"
FIGURES_DIR = RUN_ROOT / "figures"
MODELS_DIR = RUN_ROOT / "models"

RE_EXPERIMENT = "bert_generic_neg5_unweighted"
RANDOM_SEED = 42
VALIDATION_BATCH_SIZE = 16
CANDIDATE_CHUNK_SIZE = 100_000

# The requested 0.80-0.95 range selected its upper boundary. The cached
# validation probabilities therefore support a no-inference extension to
# 0.999, with 1.0 included as a zero-positive control point.
THRESHOLD_GRID = [
    0.80,
    0.825,
    0.85,
    0.875,
    0.90,
    0.925,
    0.95,
    0.96,
    0.97,
    0.98,
    0.99,
    0.995,
    0.999,
    1.0,
]
SELECTION_METRIC = "positive_macro_f1"
APPLY_SELECTED_THRESHOLD_TO_CANONICAL = True
REUSE_VALIDATION_CACHE = True

NER_JSONL = INTERIM_DIR / "ner_dataset.jsonl"
RE_CANDIDATES_CSV = INTERIM_DIR / "re_candidate_pool.csv"
LABEL_MAPS_JSON = INTERIM_DIR / "label_maps.json"
RE_RUN_CONFIG_PATH = RESULTS_DIR / f"re_{RE_EXPERIMENT}_run_config.json"
RE_MODEL_DIR = MODELS_DIR / f"re_{RE_EXPERIMENT}" / "best_model"
TEST_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / f"re_{RE_EXPERIMENT}_test_predictions.csv"
)
CANONICAL_THRESHOLD_PATH = (
    RESULTS_DIR / f"re_{RE_EXPERIMENT}_selected_threshold.json"
)
CANONICAL_THRESHOLD_SEARCH_PATH = (
    RESULTS_DIR / f"re_{RE_EXPERIMENT}_threshold_search.csv"
)
CANONICAL_TEST_METRICS_PATH = (
    RESULTS_DIR / f"re_{RE_EXPERIMENT}_thresholded_metrics.json"
)
CANONICAL_REPORT_PATH = (
    RESULTS_DIR
    / f"re_{RE_EXPERIMENT}_thresholded_classification_report.txt"
)

CACHE_PATH = (
    PREDICTIONS_DIR / f"re_{RE_EXPERIMENT}_validation_probabilities.npz"
)
CACHE_META_PATH = (
    PREDICTIONS_DIR / f"re_{RE_EXPERIMENT}_validation_probabilities_meta.json"
)
SENSITIVITY_CSV_PATH = (
    RESULTS_DIR / f"re_{RE_EXPERIMENT}_threshold_sensitivity_080_100.csv"
)
SENSITIVITY_SELECTED_PATH = (
    RESULTS_DIR / f"re_{RE_EXPERIMENT}_threshold_sensitivity_selected.json"
)
SENSITIVITY_TEST_METRICS_PATH = (
    RESULTS_DIR / f"re_{RE_EXPERIMENT}_threshold_sensitivity_test_metrics.json"
)
SENSITIVITY_REPORT_PATH = (
    RESULTS_DIR
    / f"re_{RE_EXPERIMENT}_threshold_sensitivity_classification_report.txt"
)
SENSITIVITY_FIGURE_PATH = (
    FIGURES_DIR / f"re_{RE_EXPERIMENT}_threshold_sensitivity_validation.png"
)

for path in [RESULTS_DIR, PREDICTIONS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

required_paths = [
    NER_JSONL,
    RE_CANDIDATES_CSV,
    LABEL_MAPS_JSON,
    RE_RUN_CONFIG_PATH,
    RE_MODEL_DIR,
    TEST_PREDICTIONS_PATH,
    CANONICAL_THRESHOLD_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, "Missing threshold-analysis inputs:\n- " + "\n- ".join(
    missing_paths
)

re_run_config = json.loads(RE_RUN_CONFIG_PATH.read_text(encoding="utf-8"))
label_maps = json.loads(LABEL_MAPS_JSON.read_text(encoding="utf-8"))
label_to_id = {
    label: int(index)
    for label, index in label_maps["relation_label_to_id"].items()
}
id_to_label = {
    int(index): label
    for index, label in label_maps["relation_id_to_label"].items()
}
label_names = [id_to_label[index] for index in sorted(id_to_label)]
no_relation_id = label_to_id["no_relation"]
positive_label_ids = np.array(
    [
        index
        for index, label in sorted(id_to_label.items())
        if label != "no_relation"
    ],
    dtype=int,
)

MARKER_DESIGN = re_run_config["marker_design"]
SPECIAL_TOKENS = re_run_config["special_tokens"]
MAX_LENGTH = int(re_run_config["max_length"])
CONTEXT_WINDOW = int(re_run_config["context_window"])
EXPECTED_VALIDATION_SAMPLES = int(
    re_run_config["sample_sizes"]["validation"]
)

set_seed(RANDOM_SEED)
print(
    {
        "project_root": str(PROJECT_ROOT),
        "re_experiment": RE_EXPERIMENT,
        "threshold_grid": THRESHOLD_GRID,
        "selection_metric": SELECTION_METRIC,
        "expected_validation_samples": EXPECTED_VALIDATION_SAMPLES,
        "validation_cache": str(CACHE_PATH),
        "cuda_available": torch.cuda.is_available(),
        "device": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "cpu"
        ),
    }
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select the project's .venv as the kernel."
    )


In [ ]:
expected_cache_meta = {
    "re_experiment": RE_EXPERIMENT,
    "model_checkpoint": re_run_config["model_checkpoint"],
    "random_seed": RANDOM_SEED,
    "validation_samples": EXPECTED_VALIDATION_SAMPLES,
    "max_length": MAX_LENGTH,
    "context_window": CONTEXT_WINDOW,
    "marker_design": MARKER_DESIGN,
    "label_names": label_names,
}

cache_ready = False
if REUSE_VALIDATION_CACHE and CACHE_PATH.exists() and CACHE_META_PATH.exists():
    cached_meta = json.loads(CACHE_META_PATH.read_text(encoding="utf-8"))
    cache_ready = all(
        cached_meta.get(key) == value
        for key, value in expected_cache_meta.items()
    )

print(
    {
        "reuse_validation_cache": REUSE_VALIDATION_CACHE,
        "cache_ready": cache_ready,
    }
)

tokenizer = AutoTokenizer.from_pretrained(RE_MODEL_DIR, use_fast=True)
marker_token_ids = set(tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS))
marker_id_pairs = {}
for opening_token in [
    token for token in SPECIAL_TOKENS if not token.startswith("[/")
]:
    closing_token = "[/" + opening_token[1:]
    marker_id_pairs[tokenizer.convert_tokens_to_ids(opening_token)] = (
        tokenizer.convert_tokens_to_ids(closing_token)
    )


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def entity_marker_type(label: str) -> str:
    normalised = str(label).strip().lower()
    if normalised.startswith("anatomy"):
        return "ANAT"
    if normalised.startswith("observation"):
        return "OBS"
    raise ValueError(f"Unsupported RadGraph entity label: {label}")


def marker_tokens(role: str, entity_label: str) -> tuple[str, str]:
    if MARKER_DESIGN == "generic":
        return f"[{role}]", f"[/{role}]"
    entity_type = entity_marker_type(entity_label)
    return f"[{role}_{entity_type}]", f"[/{role}_{entity_type}]"


def build_marked_tokens(
    tokens: list[str],
    head_start: int,
    head_end: int,
    tail_start: int,
    tail_end: int,
    head_label: str,
    tail_label: str,
    context_window: int,
) -> list[str]:
    left = max(0, min(head_start, tail_start) - context_window)
    right = min(
        len(tokens),
        max(head_end, tail_end) + context_window + 1,
    )
    head_open, head_close = marker_tokens("HEAD", head_label)
    tail_open, tail_close = marker_tokens("TAIL", tail_label)
    marked_tokens = []
    for index in range(left, right):
        if index == head_start:
            marked_tokens.append(head_open)
        if index == tail_start:
            marked_tokens.append(tail_open)
        marked_tokens.append(tokens[index])
        if index == head_end:
            marked_tokens.append(head_close)
        if index == tail_end:
            marked_tokens.append(tail_close)
    return marked_tokens


validation_dataset = None
if not cache_ready:
    load_started = time.perf_counter()
    documents = load_jsonl(NER_JSONL)
    tokens_by_doc = {row["doc_id"]: row["tokens"] for row in documents}

    candidate_columns = [
        "doc_id",
        "split",
        "head_start",
        "head_end",
        "tail_start",
        "tail_end",
        "head_label",
        "tail_label",
        "label",
    ]
    validation_parts = []
    reader = pd.read_csv(
        RE_CANDIDATES_CSV,
        usecols=candidate_columns,
        chunksize=CANDIDATE_CHUNK_SIZE,
    )
    for chunk in tqdm(
        reader,
        desc="Loading validation RE candidates",
        unit="chunk",
    ):
        validation_chunk = chunk[chunk["split"] == "validation"]
        if not validation_chunk.empty:
            validation_parts.append(validation_chunk.copy())

    validation_frame = pd.concat(validation_parts, ignore_index=True)
    validation_frame = validation_frame.sample(
        frac=1,
        random_state=RANDOM_SEED,
    ).reset_index(drop=True)
    assert len(validation_frame) == EXPECTED_VALIDATION_SAMPLES, (
        f"Loaded {len(validation_frame):,} validation examples; "
        f"expected {EXPECTED_VALIDATION_SAMPLES:,}."
    )

    marked_rows = []
    validation_labels = []
    for row in tqdm(
        validation_frame.itertuples(index=False),
        total=len(validation_frame),
        desc="Building pair-centred validation inputs",
        unit="pair",
    ):
        marked_rows.append(
            build_marked_tokens(
                tokens=tokens_by_doc[row.doc_id],
                head_start=int(row.head_start),
                head_end=int(row.head_end),
                tail_start=int(row.tail_start),
                tail_end=int(row.tail_end),
                head_label=row.head_label,
                tail_label=row.tail_label,
                context_window=CONTEXT_WINDOW,
            )
        )
        validation_labels.append(label_to_id[row.label])

    validation_dataset = Dataset.from_dict(
        {
            "tokens": marked_rows,
            "labels": validation_labels,
        }
    )
    print(
        {
            "validation_examples": len(validation_dataset),
            "preparation_minutes": (
                time.perf_counter() - load_started
            ) / 60,
        }
    )

    del documents
    del tokens_by_doc
    del validation_parts
    del validation_frame
    del marked_rows
    del validation_labels
else:
    print("Using saved validation probabilities; candidate reconstruction skipped.")


In [ ]:
compaction_stats = {
    "examples": 0,
    "compacted_examples": 0,
    "maximum_uncompacted_length": 0,
}


def marker_intervals(input_ids: list[int]) -> list[tuple[int, int]]:
    intervals = []
    for opening_id, closing_id in marker_id_pairs.items():
        if opening_id not in input_ids:
            continue
        opening_index = input_ids.index(opening_id)
        closing_index = input_ids.index(closing_id, opening_index + 1)
        intervals.append((opening_index, closing_index))
    intervals.sort()
    if len(intervals) != 2:
        raise ValueError(
            f"Expected two marked entity spans; found {len(intervals)}."
        )
    return intervals


def compact_pair_input(input_ids: list[int]) -> list[int]:
    compaction_stats["examples"] += 1
    compaction_stats["maximum_uncompacted_length"] = max(
        compaction_stats["maximum_uncompacted_length"],
        len(input_ids),
    )
    marker_count = sum(
        int(token_id in marker_token_ids) for token_id in input_ids
    )
    if marker_count != 4:
        raise ValueError(
            f"Expected four RE markers before compaction; found {marker_count}."
        )
    if len(input_ids) <= MAX_LENGTH:
        return input_ids

    compaction_stats["compacted_examples"] += 1
    body = input_ids[1:-1]
    intervals = marker_intervals(body)
    body_budget = MAX_LENGTH - 3
    required_indices = {
        index
        for start, end in intervals
        for index in range(start, end + 1)
    }
    if len(required_indices) > body_budget:
        raise ValueError("Marked entity spans exceed the input budget.")

    def distance_to_entity(index: int) -> int:
        distances = []
        for start, end in intervals:
            if index < start:
                distances.append(start - index)
            elif index > end:
                distances.append(index - end)
            else:
                distances.append(0)
        return min(distances)

    context_candidates = sorted(
        (
            (distance_to_entity(index), index)
            for index in range(len(body))
            if index not in required_indices
        ),
        key=lambda item: (item[0], item[1]),
    )
    selected_indices = set(required_indices)
    remaining_budget = body_budget - len(required_indices)
    selected_indices.update(
        index for _, index in context_candidates[:remaining_budget]
    )

    ordered_indices = sorted(selected_indices)
    segments = []
    segment = [body[ordered_indices[0]]]
    previous_index = ordered_indices[0]
    for index in ordered_indices[1:]:
        if index != previous_index + 1:
            segments.append(segment)
            segment = []
        segment.append(body[index])
        previous_index = index
    segments.append(segment)
    if len(segments) > 2:
        raise ValueError("Pair compaction produced too many fragments.")

    compacted_ids = [tokenizer.cls_token_id]
    for segment_index, segment_ids in enumerate(segments):
        if segment_index:
            compacted_ids.append(tokenizer.sep_token_id)
        compacted_ids.extend(segment_ids)
    compacted_ids.append(tokenizer.sep_token_id)

    compacted_marker_count = sum(
        int(token_id in marker_token_ids)
        for token_id in compacted_ids
    )
    if len(compacted_ids) > MAX_LENGTH or compacted_marker_count != 4:
        raise ValueError("Invalid compacted RE input.")
    return compacted_ids


def tokenize_batch(batch: dict) -> dict:
    raw_tokenized = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=False,
    )
    compacted_input_ids = [
        compact_pair_input(list(input_ids))
        for input_ids in raw_tokenized["input_ids"]
    ]
    tokenized = {
        "input_ids": compacted_input_ids,
        "attention_mask": [
            [1] * len(input_ids)
            for input_ids in compacted_input_ids
        ],
        "labels": batch["labels"],
    }
    if "token_type_ids" in raw_tokenized:
        tokenized["token_type_ids"] = [
            [0] * len(input_ids)
            for input_ids in compacted_input_ids
        ]
    return tokenized


tokenized_validation_dataset = None
if not cache_ready:
    tokenized_validation_dataset = validation_dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=validation_dataset.column_names,
        desc="Tokenising validation RE candidates",
        load_from_cache_file=False,
    )
    print(tokenized_validation_dataset)
    print("Validation pair compaction audit:", compaction_stats)
else:
    print("Tokenisation skipped because the validation cache is valid.")


In [ ]:
if not cache_ready:
    inference_started = time.perf_counter()
    model = AutoModelForSequenceClassification.from_pretrained(RE_MODEL_DIR)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    inference_args = TrainingArguments(
        output_dir=str(MODELS_DIR / "_threshold_sensitivity_inference"),
        per_device_eval_batch_size=VALIDATION_BATCH_SIZE,
        report_to=[],
        disable_tqdm=False,
        dataloader_num_workers=0,
        eval_accumulation_steps=32,
        fp16=torch.cuda.is_available(),
        seed=RANDOM_SEED,
    )
    trainer_kwargs = {
        "model": model,
        "args": inference_args,
        "data_collator": data_collator,
    }
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
    inference_trainer = Trainer(**trainer_kwargs)

    print("Starting validation-only RE inference")
    prediction_output = inference_trainer.predict(
        tokenized_validation_dataset
    )
    validation_logits = prediction_output.predictions
    validation_gold_ids = prediction_output.label_ids.astype(np.int64)
    validation_probabilities = torch.softmax(
        torch.tensor(validation_logits),
        dim=-1,
    ).numpy().astype(np.float32)

    np.savez_compressed(
        CACHE_PATH,
        gold_ids=validation_gold_ids,
        probabilities=validation_probabilities,
    )
    cache_meta = {
        **expected_cache_meta,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "probability_shape": list(validation_probabilities.shape),
        "pair_compaction_audit": compaction_stats,
        "inference_runtime_seconds": (
            time.perf_counter() - inference_started
        ),
    }
    CACHE_META_PATH.write_text(
        json.dumps(cache_meta, indent=2),
        encoding="utf-8",
    )
    cache_ready = True
    print(
        {
            "saved_validation_cache": str(CACHE_PATH),
            "runtime_minutes": (
                time.perf_counter() - inference_started
            ) / 60,
            "shape": validation_probabilities.shape,
        }
    )
else:
    print("Validation inference skipped; using cached probabilities.")


In [ ]:
cache = np.load(CACHE_PATH)
validation_gold_ids = cache["gold_ids"].astype(np.int64)
validation_probabilities = cache["probabilities"].astype(np.float32)
assert len(validation_gold_ids) == EXPECTED_VALIDATION_SAMPLES
assert validation_probabilities.shape == (
    EXPECTED_VALIDATION_SAMPLES,
    len(label_names),
)


def threshold_predictions(
    probabilities: np.ndarray,
    threshold: float,
) -> np.ndarray:
    positive_probabilities = probabilities[:, positive_label_ids]
    best_positive_offsets = positive_probabilities.argmax(axis=1)
    best_positive_ids = positive_label_ids[best_positive_offsets]
    best_positive_scores = positive_probabilities.max(axis=1)
    return np.where(
        best_positive_scores >= threshold,
        best_positive_ids,
        no_relation_id,
    )


def positive_metrics(
    gold_ids: np.ndarray,
    predicted_ids: np.ndarray,
) -> dict:
    micro_precision, micro_recall, micro_f1, _ = (
        precision_recall_fscore_support(
            gold_ids,
            predicted_ids,
            labels=positive_label_ids,
            average="micro",
            zero_division=0,
        )
    )
    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            gold_ids,
            predicted_ids,
            labels=positive_label_ids,
            average="macro",
            zero_division=0,
        )
    )
    return {
        "positive_micro_precision": float(micro_precision),
        "positive_micro_recall": float(micro_recall),
        "positive_micro_f1": float(micro_f1),
        "positive_macro_precision": float(macro_precision),
        "positive_macro_recall": float(macro_recall),
        "positive_macro_f1": float(macro_f1),
    }


threshold_rows = []
for threshold in THRESHOLD_GRID:
    predicted_ids = threshold_predictions(
        validation_probabilities,
        threshold,
    )
    threshold_rows.append(
        {
            "threshold": float(threshold),
            **positive_metrics(validation_gold_ids, predicted_ids),
            "predicted_positive_relations": int(
                np.isin(predicted_ids, positive_label_ids).sum()
            ),
        }
    )

validation_sensitivity = pd.DataFrame(threshold_rows)
ranked_thresholds = validation_sensitivity.sort_values(
    [
        SELECTION_METRIC,
        "positive_micro_f1",
        "positive_micro_precision",
    ],
    ascending=False,
)
selected_threshold = float(ranked_thresholds.iloc[0]["threshold"])
best_validation_metrics = json.loads(
    ranked_thresholds.iloc[0].to_json()
)

validation_sensitivity.to_csv(SENSITIVITY_CSV_PATH, index=False)

# Confirm that the new inference reproduces the previously recorded 0.80 result.
previous_search = pd.read_csv(CANONICAL_THRESHOLD_SEARCH_PATH)
previous_080 = previous_search[
    np.isclose(previous_search["threshold"], 0.80)
].iloc[-1]
current_080 = validation_sensitivity[
    np.isclose(validation_sensitivity["threshold"], 0.80)
].iloc[-1]
reproduction_differences = {
    metric: float(current_080[metric] - previous_080[metric])
    for metric in [
        "positive_micro_precision",
        "positive_micro_recall",
        "positive_micro_f1",
        "positive_macro_f1",
    ]
}
if max(abs(value) for value in reproduction_differences.values()) > 0.002:
    raise ValueError(
        "Validation inference did not reproduce the previous 0.80 metrics: "
        f"{reproduction_differences}"
    )

figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(
    validation_sensitivity["threshold"],
    validation_sensitivity["positive_micro_f1"],
    marker="o",
    label="Positive micro F1",
)
axis.plot(
    validation_sensitivity["threshold"],
    validation_sensitivity["positive_macro_f1"],
    marker="s",
    label="Positive macro F1",
)
axis.axvline(
    selected_threshold,
    color="#B33A3A",
    linestyle="--",
    label=f"Selected threshold: {selected_threshold:.3f}",
)
axis.set_xlabel("Positive-relation decision threshold")
axis.set_ylabel("Validation F1")
axis.set_title("Validation Threshold Sensitivity")
axis.set_ylim(0, 1)
axis.legend()
figure.tight_layout()
figure.savefig(SENSITIVITY_FIGURE_PATH, dpi=200)
plt.show()

display(validation_sensitivity)
print(
    {
        "selected_threshold": selected_threshold,
        "selection_metric": SELECTION_METRIC,
        "best_validation_metrics": best_validation_metrics,
        "reproduction_differences_at_080": reproduction_differences,
        "saved_table": str(SENSITIVITY_CSV_PATH),
        "saved_figure": str(SENSITIVITY_FIGURE_PATH),
    }
)


In [ ]:
test_probability_columns = [
    f"probability_{label}" for label in label_names
]
test_probability_frame = pd.read_csv(
    TEST_PREDICTIONS_PATH,
    usecols=["gold_label", *test_probability_columns],
)
test_gold_ids = test_probability_frame["gold_label"].map(
    label_to_id
).to_numpy(dtype=np.int64)
if np.isnan(test_gold_ids.astype(float)).any():
    raise ValueError("Test predictions contain an unknown gold relation label.")
test_probabilities = test_probability_frame[
    test_probability_columns
].to_numpy(dtype=np.float32)
test_predicted_ids = threshold_predictions(
    test_probabilities,
    selected_threshold,
)

test_metrics = {
    **positive_metrics(test_gold_ids, test_predicted_ids),
    "selected_positive_threshold": selected_threshold,
    "accuracy": float(
        accuracy_score(test_gold_ids, test_predicted_ids)
    ),
    "gold_positive_relations": int(
        np.isin(test_gold_ids, positive_label_ids).sum()
    ),
    "predicted_positive_relations": int(
        np.isin(test_predicted_ids, positive_label_ids).sum()
    ),
}

test_report = classification_report(
    test_gold_ids,
    test_predicted_ids,
    labels=list(range(len(label_names))),
    target_names=label_names,
    digits=4,
    zero_division=0,
)

sensitivity_selection = {
    "re_experiment_name": RE_EXPERIMENT,
    "selection_split": "validation",
    "selection_metric": SELECTION_METRIC,
    "threshold_grid": THRESHOLD_GRID,
    "selected_positive_threshold": selected_threshold,
    "random_seed": RANDOM_SEED,
    "best_validation_metrics": best_validation_metrics,
    "validation_cache": str(CACHE_PATH),
    "reproduction_differences_at_080": reproduction_differences,
    "test_metrics": test_metrics,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
SENSITIVITY_SELECTED_PATH.write_text(
    json.dumps(sensitivity_selection, indent=2),
    encoding="utf-8",
)
SENSITIVITY_TEST_METRICS_PATH.write_text(
    json.dumps(test_metrics, indent=2),
    encoding="utf-8",
)
SENSITIVITY_REPORT_PATH.write_text(test_report, encoding="utf-8")

current_threshold_config = json.loads(
    CANONICAL_THRESHOLD_PATH.read_text(encoding="utf-8")
)
previous_selected_threshold = float(
    current_threshold_config["selected_positive_threshold"]
)
threshold_changed = not np.isclose(
    selected_threshold,
    previous_selected_threshold,
)

history_dir = None
if APPLY_SELECTED_THRESHOLD_TO_CANONICAL:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    history_dir = RESULTS_DIR / "threshold_history" / run_id
    history_dir.mkdir(parents=True, exist_ok=False)
    for path in [
        CANONICAL_THRESHOLD_PATH,
        CANONICAL_THRESHOLD_SEARCH_PATH,
        CANONICAL_TEST_METRICS_PATH,
        CANONICAL_REPORT_PATH,
        RE_RUN_CONFIG_PATH,
    ]:
        if path.exists():
            shutil.copy2(path, history_dir / path.name)

    combined_search = pd.concat(
        [previous_search, validation_sensitivity],
        ignore_index=True,
    )
    combined_search = (
        combined_search.sort_values("threshold")
        .drop_duplicates(subset=["threshold"], keep="last")
    )
    combined_search.to_csv(
        CANONICAL_THRESHOLD_SEARCH_PATH,
        index=False,
    )

    canonical_threshold_config = {
        "re_experiment_name": RE_EXPERIMENT,
        "selection_split": "validation",
        "selection_metric": SELECTION_METRIC,
        "selected_positive_threshold": selected_threshold,
        "random_seed": RANDOM_SEED,
        "best_validation_metrics": best_validation_metrics,
        "threshold_sensitivity_grid": THRESHOLD_GRID,
        "sensitivity_analysis_path": str(SENSITIVITY_CSV_PATH),
        "updated_at": datetime.now().isoformat(timespec="seconds"),
    }
    CANONICAL_THRESHOLD_PATH.write_text(
        json.dumps(canonical_threshold_config, indent=2),
        encoding="utf-8",
    )
    CANONICAL_TEST_METRICS_PATH.write_text(
        json.dumps(test_metrics, indent=2),
        encoding="utf-8",
    )
    CANONICAL_REPORT_PATH.write_text(test_report, encoding="utf-8")

    re_run_config["selected_positive_threshold"] = selected_threshold
    re_run_config["positive_threshold_grid"] = sorted(
        set(
            [
                float(value)
                for value in re_run_config.get(
                    "positive_threshold_grid",
                    [],
                )
            ]
            + THRESHOLD_GRID
        )
    )
    re_run_config["thresholded_test_metrics"] = test_metrics
    re_run_config["threshold_sensitivity_analysis"] = {
        "selection_split": "validation",
        "selection_metric": SELECTION_METRIC,
        "tested_thresholds": THRESHOLD_GRID,
        "selected_threshold": selected_threshold,
        "result_path": str(SENSITIVITY_CSV_PATH),
    }
    RE_RUN_CONFIG_PATH.write_text(
        json.dumps(re_run_config, indent=2),
        encoding="utf-8",
    )

    if threshold_changed:
        temporary_prediction_path = TEST_PREDICTIONS_PATH.with_name(
            TEST_PREDICTIONS_PATH.stem + ".threshold_update.tmp.csv"
        )
        if temporary_prediction_path.exists():
            temporary_prediction_path.unlink()

        first_chunk = True
        reader = pd.read_csv(
            TEST_PREDICTIONS_PATH,
            chunksize=100_000,
        )
        for chunk in tqdm(
            reader,
            desc="Updating canonical test decision columns",
            unit="chunk",
        ):
            chunk_probabilities = chunk[
                test_probability_columns
            ].to_numpy(dtype=np.float32)
            chunk_predicted_ids = threshold_predictions(
                chunk_probabilities,
                selected_threshold,
            )
            chunk["predicted_label"] = [
                id_to_label[int(index)]
                for index in chunk_predicted_ids
            ]
            chunk["selected_positive_threshold"] = selected_threshold
            chunk["is_correct"] = (
                chunk["gold_label"] == chunk["predicted_label"]
            )
            chunk.to_csv(
                temporary_prediction_path,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                index=False,
            )
            first_chunk = False
        os.replace(temporary_prediction_path, TEST_PREDICTIONS_PATH)

    history_note = {
        "previous_selected_threshold": previous_selected_threshold,
        "new_selected_threshold": selected_threshold,
        "threshold_changed": threshold_changed,
        "canonical_prediction_file_updated": threshold_changed,
        "note": (
            "The test probability columns are unchanged and can reproduce "
            "the previous threshold decisions."
        ),
    }
    (history_dir / "THRESHOLD_UPDATE.json").write_text(
        json.dumps(history_note, indent=2),
        encoding="utf-8",
    )

print(json.dumps(test_metrics, indent=2))
print(test_report)
print(
    {
        "previous_selected_threshold": previous_selected_threshold,
        "selected_threshold": selected_threshold,
        "threshold_changed": threshold_changed,
        "canonical_outputs_updated": APPLY_SELECTED_THRESHOLD_TO_CANONICAL,
        "history_directory": (
            str(history_dir) if history_dir is not None else None
        ),
        "rerun_notebooks_07_08": threshold_changed,
    }
)


In [ ]:
result_summary = {
    "re_experiment": RE_EXPERIMENT,
    "selection_split": "validation",
    "selection_metric": SELECTION_METRIC,
    "tested_thresholds": THRESHOLD_GRID,
    "previous_threshold": previous_selected_threshold,
    "selected_threshold": selected_threshold,
    "threshold_changed": threshold_changed,
    "validation_best": best_validation_metrics,
    "test_metrics": test_metrics,
    "validation_cache": str(CACHE_PATH),
    "sensitivity_table": str(SENSITIVITY_CSV_PATH),
    "sensitivity_figure": str(SENSITIVITY_FIGURE_PATH),
    "rerun_notebooks_07_08": threshold_changed,
}

SUMMARY_PATH = (
    RESULTS_DIR / f"re_{RE_EXPERIMENT}_threshold_sensitivity_summary.json"
)
SUMMARY_PATH.write_text(
    json.dumps(result_summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(result_summary, indent=2))
print("Saved:", SUMMARY_PATH)
